# Multi-Agent Workflow: Research Agent & Plotly Chart Generator Agent

This notebook builds a multi-agent system combining:
1. **Research Agent**: Real-time web search for precise prices & statistics.
2. **Chart Generator Agent**: Python REPL code execution to create & save interactive charts using **Plotly** ().


In [10]:
# =========================
# 1. Imports & Environment Setup
# =========================
import os
import uuid
from typing import Annotated
from dotenv import load_dotenv

load_dotenv(override=True)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig
from langchain_experimental.utilities import PythonREPL
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from ddgs import DDGS
from IPython.display import display, HTML, Image

# Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
print("Environment initialized successfully!")



Environment initialized successfully!


In [11]:
# =========================
# 2. Tools Setup (Search & Python REPL)
# =========================
@tool
def web_search_tool(query: str) -> str:
    """Search the web using DuckDuckGo for real-time prices, data, and information."""
    ddg = DDGS()
    results = list(ddg.text(query, max_results=5))
    if not results:
        return "No search results found."
    formatted = []
    for r in results:
        title = r.get("title", "")
        link = r.get("href", "")
        snippet = r.get("body", "")
        formatted.append(f"Title: {title}\nLink: {link}\nSnippet: {snippet}\n")
    return "\n".join(formatted)

python_repl = PythonREPL()

@tool
def python_repl_tool(
    code: str
) -> str:
    """Executes Python code to generate and save interactive Plotly charts.

    Args:
        code: Pure Python code string using Plotly. Must save the interactive figure to chart.html using fig.write_html('chart.html').
    """
    try:
        result = python_repl.run(code)
    except Exception as e:
        return f"Execution failed: {repr(e)}"
    return f"Python executed successfully.\n{result}"



In [12]:
# =========================
# 3. Test Search Tool
# =========================
res = web_search_tool.invoke({"query": "buy iPhone 15 128GB price Flipkart Amazon India"})
print(res[:300])



Title: Planning to buy iPhone 15 128GB ahead of iPhone 17 launch? Check sale price comparison on Amazon and Flipkart - Technology News | The Financial Express
Link: https://www.financialexpress.com/life/technology-planning-to-buy-iphone-15-128gb-ahead-of-iphone-17-launch-check-sale-price-comparison-


In [13]:
# =========================
# 4. Helper Function for System Prompts
# =========================
def make_system_prompt(suffix: str) -> str:
    return (
        "You are part of a multi-agent system.\n"
        "Follow your role strictly.\n"
        "Do not refuse tasks.\n"
        "Do not explain limitations.\n\n"
        + suffix
    )



In [14]:
# =========================
# 5. Research Agent
# =========================
research_agent_system_prompt = """You are a strict, factual research agent.

Your job is to gather accurate real-time data needed for chart generation.

STRICT RULES:
1. Use web_search_tool to search specifically for each requested platform or category.
2. For e-commerce price comparisons (e.g. Amazon India, Flipkart, Meesho):
   - Search specifically for queries like 'buy iPhone 15 128GB price Flipkart Amazon India' or 'site:amazon.in iPhone 15 128GB price'.
   - Extract standard retail selling prices in INR (e.g., ~₹54,900 - ₹60,999).
   - CRITICAL MEESHO RULE: Meesho is an Indian e-commerce platform that sells ONLY phone covers/cases/accessories (under ₹300). Meesho does NOT sell official new Apple iPhones. You MUST report Meesho price as 'N/A (Accessories only)'. DO NOT hallucinate a full iPhone price for Meesho (such as 48000 or 49000).
3. ALWAYS extract exact real numeric values for platforms that sell the product.
4. DO NOT generate charts or python code yourself.

OUTPUT FORMAT (MANDATORY):
Return extracted data in a clean structured format:
Label: <store or category name>
Value: <numeric price or N/A>
Unit: <currency or note>"""

research_agent = create_react_agent(
    llm,
    tools=[web_search_tool],
    prompt=make_system_prompt(research_agent_system_prompt),
)

def research_node(state: MessagesState) -> MessagesState:
    result = research_agent.invoke(state)
    return {"messages": result["messages"]}



/var/folders/ry/0dp2gqbj4nv8q6qbdj7z67700000gn/T/ipykernel_16134/23661053.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(


In [15]:
# =========================
# 6. Chart Generator Agent (Plotly)
# =========================
chart_generator_system_prompt = """You are a chart-generation agent.

You MUST call python_repl_tool to generate and save an interactive Plotly chart using Python code.

STRICT CODE INSTRUCTIONS:
1. Strictly use Plotly (import plotly.express as px or import plotly.graph_objects as go) for chart generation.
2. For price comparisons across stores/platforms, ALWAYS use a Plotly Bar Chart (px.bar). Do NOT use a pie chart for price comparisons.
3. Parse numeric values carefully:
   - For platforms with valid numeric prices (e.g., Amazon India: 60999, Flipkart: 59900), plot vertical bars and show text price labels on top of bars.
   - For platforms with 'N/A' or 'Accessories only' (such as Meesho), set height to 0 or exclude, and add a text annotation 'N/A (Accessories Only)'.
4. Set Title ('iPhone 15 128GB Price Comparison'), X-axis ('Platform'), and Y-axis ('Price in INR ₹').
5. Save the interactive chart as an HTML file named 'chart.html' using fig.write_html('chart.html').
   Also attempt static PNG export: try: fig.write_image('chart.png') except Exception: pass
6. ALWAYS call python_repl_tool with your Python code.

After successful tool execution, summarize the chart findings cleanly."""

chart_agent = create_react_agent(
    llm,
    tools=[python_repl_tool],
    prompt=make_system_prompt(chart_generator_system_prompt),
)

def chart_node(state: MessagesState) -> MessagesState:
    result = chart_agent.invoke(state)
    return {"messages": result["messages"]}



/var/folders/ry/0dp2gqbj4nv8q6qbdj7z67700000gn/T/ipykernel_16134/253087400.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  chart_agent = create_react_agent(


In [16]:
# =========================
# 7. Multi-Agent Graph Construction
# =========================
workflow = StateGraph(MessagesState)
workflow.add_node("researcher", research_node)
workflow.add_node("chart_generator", chart_node)

workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "chart_generator")
workflow.add_edge("chart_generator", END)

app = workflow.compile(checkpointer=MemorySaver())
print("Graph compiled successfully!")



Graph compiled successfully!


In [ ]:
# =========================
# 8. Execution Example 1: GDP Comparison
# =========================
config = RunnableConfig(
    recursion_limit=20,
    configurable={"thread_id": str(uuid.uuid4())}
)

inputs = {
    "messages": [
        HumanMessage(
            content="do a research on gdps of india, usa, and china for the year 2026 & then compare them using a Plotly pie chart"
        )
    ]
}

result = app.invoke(inputs, config=config)
print("Final Answer:\n", result["messages"][-1].content)





Final Answer:
 


In [19]:
# Display generated interactive Plotly chart directly in notebook cell output
if os.path.exists("chart.html"):
    display(HTML(filename="chart.html"))
elif os.path.exists("chart.png"):
    display(Image("chart.png"))

In [ ]:
# =========================
# 9. Execution Example 2: E-commerce Price Comparison
# =========================
config2 = RunnableConfig(
    recursion_limit=20,
    configurable={"thread_id": str(uuid.uuid4())}
)

inputs2 = {
    "messages": [
        HumanMessage(
            content="Compare prices of iPhone 15 128GB on Amazon India, Flipkart, Meesho & then plot a Plotly bar chart for the same"
        )
    ]
}

result2 = app.invoke(inputs2, config=config2)
print("Final Answer:\n", result2["messages"][-1].content)





Final Answer:
 


In [20]:
# Display generated interactive Plotly chart directly in notebook cell output
if os.path.exists("chart.html"):
    display(HTML(filename="chart.html"))
elif os.path.exists("chart.png"):
    display(Image("chart.png"))